In [ ]:
!pip install scikit-learn -q

import os
import numpy as np
import torch
from PIL import Image, ImageOps
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from sklearn.model_selection import StratifiedShuffleSplit
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DRIVE_ROOT    = "/content/drive/MyDrive/AML_Dataset/Warp-C"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class PadToSquare:
    def __call__(self, img):
        w, h = img.size
        d = max(w, h)
        pl, pt = (d - w) // 2, (d - h) // 2
        return ImageOps.expand(img, (pl, pt, d - w - pl, d - h - pt), fill=(128, 128, 128))

train_pipeline = transforms.Compose([
    PadToSquare(),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.08),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

eval_pipeline = transforms.Compose([
    PadToSquare(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
def get_dataloaders(root=DRIVE_ROOT, batch_size=32, val_split=0.2, num_workers=2, seed=42):
    torch.manual_seed(seed)

    train_dir = f"{root}/train_crops"
    test_dir  = f"{root}/test_crops"

    def make_flat_dataset(root_dir, transform):
        samples, class_to_idx = [], {}

        for superclass in sorted(os.listdir(root_dir)):
            superclass_path = os.path.join(root_dir, superclass)
            if not os.path.isdir(superclass_path):
                continue

            for subclass in sorted(os.listdir(superclass_path)):
                subclass_path = os.path.join(superclass_path, subclass)
                if not os.path.isdir(subclass_path):
                    continue

                images_here = [f for f in os.listdir(subclass_path) if f.lower().endswith(".jpg")]

                if images_here:
                    # bottle, cardboard, detergent: superclass/subclass/images
                    if subclass not in class_to_idx:
                        class_to_idx[subclass] = len(class_to_idx)
                    for img_name in images_here:
                        samples.append((os.path.join(subclass_path, img_name), class_to_idx[subclass]))
                else:
                    # canister, cans: superclass/subclass/subsubclass/images
                    for subsubclass in sorted(os.listdir(subclass_path)):
                        subsubclass_path = os.path.join(subclass_path, subsubclass)
                        if not os.path.isdir(subsubclass_path):
                            continue
                        if subsubclass not in class_to_idx:
                            class_to_idx[subsubclass] = len(class_to_idx)
                        for img_name in os.listdir(subsubclass_path):
                            if img_name.lower().endswith(".jpg"):
                                samples.append((os.path.join(subsubclass_path, img_name), class_to_idx[subsubclass]))

        # Build a dummy ImageFolder then override its internals
        dataset = datasets.ImageFolder(root_dir, transform=transform)
        dataset.samples      = samples
        dataset.imgs         = samples
        dataset.targets      = [s[1] for s in samples]
        dataset.classes      = list(class_to_idx.keys())
        dataset.class_to_idx = class_to_idx
        return dataset

    train_ds_aug  = make_flat_dataset(train_dir, transform=train_pipeline)
    train_ds_eval = make_flat_dataset(train_dir, transform=eval_pipeline)
    test_ds       = make_flat_dataset(test_dir,  transform=eval_pipeline)

    print(f"Total train images : {len(train_ds_aug.samples)}")
    print(f"Total test images  : {len(test_ds.samples)}")
    print(f"Classes ({len(train_ds_aug.classes)}): {train_ds_aug.classes}")

    labels = np.array(train_ds_aug.targets)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_split, random_state=seed)
    train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))

    train_labels   = labels[train_idx]
    class_counts   = np.bincount(train_labels)
    sample_weights = torch.from_numpy((1.0 / class_counts[train_labels]).astype(np.float64))
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(Subset(train_ds_aug,  train_idx), batch_size=batch_size,
                              sampler=sampler, num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(Subset(train_ds_eval, val_idx),   batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)

    print(f"\nTrain: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader, train_ds_aug, train_ds_eval, test_ds, train_idx, val_idx

In [ ]:
train_loader, val_loader, test_loader, train_ds_aug, train_ds_eval, test_ds, train_idx, val_idx = get_dataloaders()

images, labels = next(iter(train_loader))
print(images.shape)  # → torch.Size([32, 3, 224, 224])
print(labels.shape)  # → torch.Size([32])

Total train images : 8823
Total test images  : 1551
Classes (28): ['bottle-blue', 'bottle-blue-full', 'bottle-blue5l', 'bottle-blue5l-full', 'bottle-dark', 'bottle-dark-full', 'bottle-green', 'bottle-green-full', 'bottle-milk', 'bottle-milk-full', 'bottle-multicolor', 'bottle-multicolorv-full', 'bottle-oil', 'bottle-oil-full', 'bottle-transp', 'bottle-transp-full', 'bottle-yogurt', 'glass-dark', 'glass-green', 'glass-transp', 'canister', 'cans', 'juice-cardboard', 'milk-cardboard', 'detergent-box', 'detergent-color', 'detergent-transparent', 'detergent-white']

Train: 7058 | Val: 1765 | Test: 1551


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([32, 3, 224, 224])
torch.Size([32])


In [ ]:
import os
from PIL import Image

SAVE_DIR = "/content/WaRP-C-preprocessed"

def build_superclass_map(crops_dir):
    """Automatically builds subclass → superclass mapping from folder structure."""
    mapping = {}
    for superclass in os.listdir(crops_dir):
        superclass_path = os.path.join(crops_dir, superclass)
        if not os.path.isdir(superclass_path):
            continue
        for subclass in os.listdir(superclass_path):
            subclass_path = os.path.join(superclass_path, subclass)
            if os.path.isdir(subclass_path):
                mapping[subclass] = superclass
    return mapping

superclass_map = build_superclass_map(f"{DRIVE_ROOT}/train_crops")
print("Superclass map:", superclass_map)

# train and val
for split, dataset, indices in [("train", train_ds_aug,  train_idx),
                                  ("val",   train_ds_eval, val_idx)]:
    class_names = dataset.classes
    print(f"\nSaving {split}...")
    count = 0

    for idx in indices:
        img_path, label = dataset.samples[idx]
        cls_name   = class_names[label]
        superclass = superclass_map.get(cls_name, "other")
        out_dir    = f"{SAVE_DIR}/{split}/{superclass}/{cls_name}"
        os.makedirs(out_dir, exist_ok=True)

        img = Image.open(img_path).convert("RGB")
        img = PadToSquare()(img)
        img = img.resize((224, 224), Image.BILINEAR)
        img.save(f"{out_dir}/{os.path.basename(img_path)}")
        count += 1

    print(f"  {split}: {count} images saved")

# test
print(f"\nSaving test...")
count = 0
for img_path, label in test_ds.samples:
    cls_name   = test_ds.classes[label]
    superclass = superclass_map.get(cls_name, "other")
    out_dir    = f"{SAVE_DIR}/test/{superclass}/{cls_name}"
    os.makedirs(out_dir, exist_ok=True)

    img = Image.open(img_path).convert("RGB")
    img = PadToSquare()(img)
    img = img.resize((224, 224), Image.BILINEAR)
    img.save(f"{out_dir}/{os.path.basename(img_path)}")
    count += 1

print(f"  test: {count} images saved")
print(f"\nAll done. Saved to {SAVE_DIR}")

Superclass map: {'detergent-white': 'detergent', 'detergent-transparent': 'detergent', 'detergent-color': 'detergent', 'detergent-box': 'detergent', 'milk-cardboard': 'cardboard', 'juice-cardboard': 'cardboard', 'glass-transp': 'bottle', 'bottle-yogurt': 'bottle', 'glass-dark': 'bottle', 'glass-green': 'bottle', 'bottle-oil': 'bottle', 'bottle-multicolorv-full': 'bottle', 'bottle-multicolor': 'bottle', 'bottle-transp': 'bottle', 'bottle-transp-full': 'bottle', 'bottle-oil-full': 'bottle', 'bottle-milk-full': 'bottle', 'bottle-milk': 'bottle', 'bottle-green-full': 'bottle', 'bottle-green': 'bottle', 'bottle-dark-full': 'bottle', 'bottle-dark': 'bottle', 'bottle-blue5l-full': 'bottle', 'bottle-blue': 'bottle', 'bottle-blue5l': 'bottle', 'bottle-blue-full': 'bottle', 'cans': 'cans', 'canister': 'canister'}

Saving train...
  train: 7058 images saved

Saving val...
  val: 1765 images saved

Saving test...
  test: 1551 images saved

All done. Saved to /content/WaRP-C-preprocessed


In [ ]:
from google.colab import drive, files
shutil.make_archive("/content/WaRP-C-preprocessed", "zip", "/content/WaRP-C-preprocessed")
files.download("/content/WaRP-C-preprocessed.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>